In [0]:
coffe = spark.read.table('data.orders.coffe_sales')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
# print sample data
coffe.limit(5).show(truncate=False)

In [0]:
# Find all transactions for a specific Weekday (e.g., 'Monday')
coffe.filter(f.col('Weekday') == 'Mon').display()


In [0]:
# count number of transactions per Month_name
coffe.groupBy('Month_name', 'Monthsort').agg(f.count('*').alias('count'))\
    .select("Month_name",'count').orderBy(f.col('monthsort')).display()

In [0]:
# Calculate total sales (money) per coffee_name
coffe.groupBy('coffee_name').agg(f.round(f.sum('money'),2).alias('total')).alias('total').display()

In [0]:
# Find average spending per cash_type
coffe.groupBy('cash_type').agg(f.round(f.avg('money'), 2).alias('avg')).display()

In [0]:
# Get top 5 highest transactions by money
coffe.orderBy(f.col('money').desc()).limit(5).display()

In [0]:
# Count transactions per hour_of_day
coffe.groupBy('hour_of_day').agg(f.count('*').alias('transaction_count')).orderBy('hour_of_day').display()

In [0]:
# Find busiest hour (highest number of transactions)
busiest_hour = coffe.groupBy('hour_of_day').agg(f.count('*').alias('transaction_count'))\
    .orderBy(f.col('transaction_count').desc()).limit(1)
display(busiest_hour)

In [0]:
# Total sales per Weekday
coffe.groupBy('Weekday').agg(f.round(f.sum('money'), 2).alias('total_sales')).orderBy('Weekday').display()

In [0]:
# Time_of_Day with highest revenue
top_time_of_day = coffe.groupBy('Time_of_Day').agg(f.round(f.sum('money'),2).alias('revenue'))\
    .orderBy(f.col('revenue').desc()).limit(1)
display(top_time_of_day)

In [0]:
# Monthly sales trend using Month_name
coffe.groupBy('Month_name', 'Monthsort').agg(f.round(f.sum('money'), 2).alias('monthly_sales'))\
    .orderBy('Monthsort').select('Month_name', 'monthly_sales').display()

In [0]:
# Number of transactions per (coffee_name, cash_type)
coffe.groupBy('coffee_name', 'cash_type').agg(f.count('*').alias('transaction_count')).display()

In [0]:
# Top-selling coffee by revenue
top_coffee = coffe.groupBy('coffee_name').agg(f.round(f.sum('money'),2).alias('revenue'))\
    .orderBy(f.col('revenue').desc()).limit(1)
display(top_coffee)

In [0]:
# Rank coffee types based on total revenue.
w = Window.orderBy(f.col('revenue').desc())
coffe.groupBy('coffee_name')\
    .agg(f.round(f.sum('money'),2).alias('revenue'))\
    .orderBy(f.col('revenue').desc())\
    .withColumn('rank', f.rank().over(w)).display()

In [0]:
# Find cumulative sales over time (using Date).
w = Window.orderBy(f.col('Date'))
coffe.groupBy('Date')\
    .agg(f.round(f.sum('money'),2).alias('daily_sales'))\
    .orderBy(f.col('Date'))\
    .withColumn('cumulative_sales',f.sum('daily_sales').over(w)).display()

In [0]:
# Calculate moving average of sales over last 3 days.
w = Window.orderBy(f.col('Date')).rowsBetween(-3, 0)
coffe.groupBy('Date')\
    .agg(f.round(f.sum('money'),2).alias('daily_sales'))\
    .orderBy(f.col('Date'))\
    .withColumn('avg', f.round(f.avg('daily_sales').over(w),2))\
    .display()

In [0]:
# Find percentage contribution of each coffee_name to total sales.
w = Window.orderBy(f.col('Date'))
coffe.groupBy('Date', 'coffee_name')\
    .agg(f.round(f.sum('money'),2).alias('daily_sales'))\
    .orderBy(f.col('Date'))\
    .withColumn('total_sales', f.sum('daily_sales').over(w))\
    .withColumn('percent', f.round(f.col('daily_sales')*100/f.col('total_sales'),2))\
    .display()

In [0]:
#Detect hours where sales are above average.
df = coffe.groupBy('hour_of_day').agg(f.round(f.sum('money'),2).alias('total'))
avg = df.agg(f.avg('total')).collect()[0][0]
result = df.filter(f.col('total') > avg).orderBy('hour_of_day')
display(result)

In [0]:
# Find most preferred payment method (cash_type) per coffee_name.
from pyspark.sql.functions import row_number
w = Window.partitionBy('coffee_name').orderBy(f.col('count').desc())
coffe.groupBy('coffee_name', 'cash_type').agg(f.count('*').alias('count'))\
    .withColumn('rn', row_number().over(w))\
    .filter(f.col('rn') == 1)\
    .select('coffee_name', 'cash_type', 'count').display()

In [0]:
# Segment customers behavior by Time_of_Day and spending pattern.
coffe.groupBy('Time_of_Day').agg(
    f.round(f.avg('money'),2).alias('avg_spending'),
    f.round(f.sum('money'),2).alias('total_spending'),
    f.count('*').alias('transaction_count')
).orderBy('Time_of_Day').display()